# Module 23 — SHAP Explainability for Fraud Decisions

> **Track 2 · Revolut Fintech Specialization**  
> **Difficulty**: Intermediate → Advanced  
> **Runtime**: ~5 minutes  
> **Key Libraries**: SHAP, Scikit-learn, Pandas, NumPy, Plotly  
> **Prerequisite**: Module 13 (Logistic Regression), Module 22 (Credit Risk Scoring)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Load Fraud Model and Data](#3-load-fraud-model-and-data)
4. [Train Fraud Model](#4-train-fraud-model)
5. [SHAP Value Computation](#5-shap-value-computation)
6. [Global Feature Importance](#6-global-feature-importance)
7. [Individual Prediction Explanations](#7-individual-prediction-explanations)
8. [Waterfall Plots](#8-waterfall-plots)
9. [Business Explanations](#9-business-explanations)
10. [Key Business Takeaway](#10-key-business-takeaway)

---
## §1 · Business Context

### Why explainability matters for fraud detection?

SHAP (SHapley Additive exPlanations) provides **transparent AI**:
- **Regulatory compliance**: Explain why transactions are blocked.
- **Customer trust**: Provide clear reasons for fraud flags.
- **Model debugging**: Understand what the model has learned.

**SHAP advantages**:
1. **Game-theoretic foundation**: Based on Shapley values from cooperative game theory.
2. **Additive explanations**: Each feature's contribution sums to the prediction.
3. **Global and local**: Understand both overall model behavior and individual predictions.

**Business applications**:
- **Fraud analysts**: Investigate flagged transactions efficiently.
- **Customer support**: Explain blocked transactions to customers.
- **Compliance**: Demonstrate model fairness and lack of bias.
- **Model development**: Identify important features and potential issues.

In this notebook we will:
1. Train a fraud detection model.
2. Compute SHAP values for predictions.
3. Create summary, force, and waterfall plots.
4. Generate human-readable explanations for compliance.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import time
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Scikit-learn ─────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.ensemble import RandomForestClassifier

# ── SHAP ─────────────────────────────────────────────────────────
import shap

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid")

print("✓ All imports loaded")

---
## §3 · Load Fraud Model and Data

In [ ]:
# Recreate the fraud model from Module 13
np.random.seed(42)

N_SAMPLES = 50000
FRAUD_RATE = 0.02

# Generate synthetic fraud data
data = {
    'amount': np.random.exponential(100, N_SAMPLES),
    'hour': np.random.randint(0, 24, N_SAMPLES),
    'day_of_week': np.random.randint(0, 7, N_SAMPLES),
    'merchant_category': np.random.choice(['grocery', 'online', 'travel', 'gas', 'restaurant'], N_SAMPLES),
    'country_risk': np.random.uniform(0, 1, N_SAMPLES),
    'distance_from_home': np.random.exponential(50, N_SAMPLES),
    'txn_velocity_1h': np.random.poisson(2, N_SAMPLES),
    'txn_velocity_24h': np.random.poisson(10, N_SAMPLES),
    'avg_amount_30d': np.random.exponential(80, N_SAMPLES),
    'std_amount_30d': np.random.exponential(30, N_SAMPLES),
}

df = pd.DataFrame(data)

# Generate fraud labels
fraud_prob = (
    0.3 * (df['amount'] > 500).astype(float)
    + 0.2 * (df['country_risk'] > 0.7).astype(float)
    + 0.2 * (df['txn_velocity_1h'] > 3).astype(float)
    + 0.1 * (df['distance_from_home'] > 100).astype(float)
    + np.random.normal(0, 0.1, N_SAMPLES)
)
fraud_prob = 1 / (1 + np.exp(-fraud_prob + 3))  # Sigmoid with low base rate
df['is_fraud'] = np.random.binomial(1, fraud_prob)

# Encode categorical
df = pd.get_dummies(df, columns=['merchant_category'], drop_first=True)

feature_cols = [c for c in df.columns if c != 'is_fraud']
X = df[feature_cols]
y = df['is_fraud']

print(f"✓ Loaded fraud data:")
print(f"  Samples: {len(df):,}")
print(f"  Fraud rate: {y.mean()*100:.2f}%")
print(f"  Features: {len(feature_cols)}")

---
## §4 · Train Fraud Model

In [ ]:
# Train a fraud detection model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
model.fit(X_train, y_train)

y_prob = model.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test, y_prob)

print(f"✓ Fraud model trained")
print(f"  ROC-AUC: {roc_auc:.4f}")

---
## §5 · SHAP Value Computation

In [ ]:
# Compute SHAP values
# Use a sample for faster computation
X_sample = X_test.sample(1000, random_state=42)

# Create SHAP explainer
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)

print(f"✓ SHAP values computed")
print(f"  Shape: {np.array(shap_values).shape}")
print(f"  Explaining {len(X_sample)} predictions")

---
## §6 · Global Feature Importance

In [ ]:
# SHAP summary plot
print("SHAP Summary Plot (Global Feature Importance):")
print("=" * 70)

# Summary plot
shap.summary_plot(shap_values[1], X_sample, plot_type="bar", show=False)
plt.title("SHAP Feature Importance (Fraud Detection)")
plt.tight_layout()
plt.show()

print("\n💡 Key insights:")
print("   - Amount is the most important feature")
print("   - Country risk and velocity features matter")
print("   - SHAP shows both magnitude and direction of impact")

---
## §7 · Individual Prediction Explanations

In [ ]:
# Explain individual predictions
print("=" * 70)
print("INDIVIDUAL PREDICTION EXPLANATIONS")
print("=" * 70)

# Find a high-risk prediction
high_risk_idx = np.argmax(y_prob)
low_risk_idx = np.argmin(y_prob)

print("\n--- High-Risk Transaction ---")
print(f"Predicted fraud probability: {y_prob[high_risk_idx]:.4f}")
shap.force_plot(
    explainer.expected_value[1],
    shap_values[1][high_risk_idx],
    X_sample.iloc[high_risk_idx],
    matplotlib=True
)
plt.show()

print("\n--- Low-Risk Transaction ---")
print(f"Predicted fraud probability: {y_prob[low_risk_idx]:.4f}")
shap.force_plot(
    explainer.expected_value[1],
    shap_values[1][low_risk_idx],
    X_sample.iloc[low_risk_idx],
    matplotlib=True
)
plt.show()

---
## §8 · Waterfall Plots

In [ ]:
# Waterfall plot for a single prediction
print("=" * 70)
print("WATERFALL PLOT - DETAILED EXPLANATION")
print("=" * 70)

# Select a transaction
idx = 0

# Create explanation object
explanation = shap.Explanation(
    values=shap_values[1][idx],
    base_values=explainer.expected_value[1],
    data=X_sample.iloc[idx].values,
    feature_names=feature_cols
)

shap.waterfall_plot(explanation, show=False)
plt.title("Waterfall Plot: How Each Feature Contributes")
plt.tight_layout()
plt.show()

print("\n💡 Waterfall plots show:")
print("   - Base value (average prediction)")
print("   - Each feature's contribution (red = increase, blue = decrease)")
print("   - Final prediction")

---
## §9 · Business Explanations

In [ ]:
# Generate human-readable explanations
def generate_explanation(shap_values, features, threshold=0.1):
    """Generate human-readable fraud explanation."""
    # Get top contributing features
    contributions = list(zip(features, shap_values))
    contributions.sort(key=lambda x: abs(x[1]), reverse=True)
    
    explanation = "Transaction flagged because:\n"
    for feat, val in contributions[:5]:
        if abs(val) > threshold:
            direction = "increases" if val > 0 else "decreases"
            explanation += f"  - {feat} {direction} fraud risk by {abs(val):.3f}\n"
    
    return explanation

# Generate explanations for sample transactions
print("=" * 70)
print("HUMAN-READABLE FRAUD EXPLANATIONS")
print("=" * 70)

for i in range(3):
    print(f"\nTransaction {i+1} (fraud prob: {y_prob[i]:.4f}):")
    print(generate_explanation(shap_values[1][i], feature_cols))

print("\n💡 These explanations can be:")
print("   - Sent to compliance teams for review")
print("   - Used in customer disputes")
print("   - Provided to fraud analysts for investigation")

---
## §10 · Key Business Takeaway

> ### 🎯 Action Items for Fraud & Compliance Teams
>
> | Aspect | Recommendation |
> |--------|---------------|
> | **When to use** | Fraud alerts, customer disputes, regulatory audits |
> | **Explanation type** | SHAP values for feature contributions |
> | **Visualization** | Summary plots for global, waterfall for local |
> | **Output format** | Human-readable text for non-technical stakeholders |
> | **Deployment** | Real-time explanations for every flagged transaction |
>
> ### 💡 Revolut-Specific Guidelines
>
> 1. **Explain every fraud flag**:
>    ```
>    Transaction flagged → Compute SHAP → Generate explanation → Send to analyst
>    ```
>
> 2. **Explanation templates**:
>    | Audience | Format | Detail Level |
>    |----------|--------|--------------|
>    | Customer | "Transaction blocked due to unusual amount" | Low |
>    | Analyst | Full SHAP waterfall plot | High |
>    | Compliance | Feature contributions + model details | Very High |
>
> 3. **Use SHAP for model fairness**:
>    - Check if protected features (age, gender) have high SHAP values
>    - Ensure model doesn't discriminate against protected groups
>    - Document fairness metrics for regulators
>
> ### 🔑 Key Takeaways
>
> 1. **SHAP provides game-theoretic guarantees** for feature importance.
> 2. **Waterfall plots** are best for explaining individual predictions.
> 3. **Summary plots** reveal global model behavior.
> 4. **Human-readable explanations** build trust with customers and regulators.
> 5. **Use SHAP for fairness audits** to detect discriminatory patterns.